##### Initial data exploration of the PAN12 dataset.
This notebook is using a Colab server and is using open source data from the PAN12 dataset extracted from https://zenodo.org/records/3713280

In [1]:
# mounting my google drive to access the data file

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# relevant imports

import xml.etree.ElementTree as ET 
import os
import pandas as pd
import re

In [ ]:
# defining the file path to the xml data file

file_path = "/content/drive/MyDrive/research/code/pan_12/pan_12_data/pan_12_train/training_corpus.xml"

In [4]:
# function to parse the xml file and convert it into a pandas dataframe

def parse_pan12_xml(file_path):
    tree = ET.parse(file_path)
    root = tree.getroot()
    # initialise a list to hold the records for each message
    records = []
    # iterate through each conversation and then through each message to extract the relevant information
    for conversation in root.findall('conversation'):
        convo_id = conversation.attrib.get('id')
        for message in conversation.findall('message'):
            author = message.findtext('author')
            time   = message.findtext('time')
            text   = message.findtext('text')
            # append the extracted information as a dictionary to the records list
            records.append({
                'conversation_id': convo_id,
                'user_id':         author,
                'time':            time,
                'message':         text
            })
    # convert the list of records into a pandas dataframe and return it
    return pd.DataFrame(records)

df = parse_pan12_xml(file_path)
# sort the dataframe by conversation_id and time to ensure the messages are in the correct order within each conversation
df_sorted = df.sort_values(by=["conversation_id", "time"], ascending=[True, True]).reset_index(drop=True)


In [17]:
# identify number of rows and columns in the dataframe

print(f"\nShape: {df_sorted.shape}")


Shape: (903607, 4)


In [18]:
# identify number of unique conversations in the dataframe

print(f"Number of unique conversations: {df_sorted['conversation_id'].nunique()}")

Number of unique conversations: 66927


In [19]:
# identify number of unique users in the dataframe

print(f"Number of unique users: {df_sorted['user_id'].nunique()}")

Number of unique users: 97689


In [20]:
# identify number of conversations with 10 or more messages

ten_or_more_messages = df_sorted.groupby('conversation_id').size()
print(f"Conversations with 10 or more messages: {(ten_or_more_messages >= 10).sum()}")

Conversations with 10 or more messages: 15550


In [21]:
# adding labels to the dataframe to identify which messages are from predators

predator_ids_path = "/content/drive/MyDrive/research/code/pan_12/pan_12_data/pan_12_train//predators_id.txt"

with open(predator_ids_path, 'r') as f:
    predator_ids = set(line.strip() for line in f)
df_sorted['is_predator'] = df_sorted['user_id'].apply(lambda x: x in predator_ids)

print(f"Number of messages from predators: {df_sorted['is_predator'].sum()}")

Number of messages from predators: 40978


In [22]:
# null counts per column

print(df_sorted.isnull().sum())
print(f"\nTotal nulls: {df_sorted.isnull().sum().sum()}")
print(f"Rows with at least one null: {df_sorted.isnull().any(axis=1).sum()}")

conversation_id    0
user_id            0
time               0
message            0
is_predator        0
dtype: int64

Total nulls: 0
Rows with at least one null: 0
